# Task 1.3 — What the Paper Claims to Improve
**Paper:** *Training SVMs Without Offset* — Steinwart, Hush & Scovel, JMLR 2011

**Student:** Kush Agarwal | **Roll No.:** 230050 | NST, Rishihood University, Sonipat

## Main Baseline Method

The paper's primary baseline is **LIBSVM** (Chang & Lin, 2001), the dominant open-source SVM library at the time of writing, which implements the standard SVM with bias using Sequential Minimal Optimization (SMO).

---

## Limitation of LIBSVM Identified by the Paper

The standard SVM dual — the one LIBSVM solves — contains the equality constraint Σ αᵢyᵢ = 0. This constraint exists because the bias b is a free primal variable; the Lagrangian stationarity condition with respect to b produces this coupling. The consequence is that no single αᵢ can be changed without simultaneously changing at least one other αⱼ to keep the sum equal to zero. Because of this structural coupling, LIBSVM must update **two** dual variables at every step. The 2D sub-problem is still a simple quadratic, but it adds complexity: the algorithm must search for a suitable pair (i, j) to update together, which requires more involved working-set selection logic (including second-order information in the improved SMO). This makes each iteration slower and the working-set selection code significantly more complex than it needs to be.

---

## How the Proposed Method Overcomes This

By dropping b from the primal, the equality constraint never appears in the dual. Each variable αᵢ is bounded only by its own individual box constraint 0 ≤ αᵢ ≤ C — there is no coupling between variables at all. This makes each coordinate independent, so the algorithm can update exactly one variable per iteration using the closed-form step αᵢ ← clip(αᵢ + gᵢ/kᵢᵢ, 0, C). No sub-problem needs to be solved, no pair needs to be selected, and the feasibility of the update is guaranteed by the clip operation alone.

---

## Scenario Where the Paper's Method Would NOT Outperform LIBSVM

The offset-free SVM would likely underperform LIBSVM in any setting where the data cannot be adequately centered before training. Consider a 2D binary classification problem where class +1 is clustered around (10, 10) and class −1 is clustered around (10, 8). The optimal separating hyperplane here is approximately horizontal at y ≈ 9, which in equation form requires w = (0, 1) and b ≈ −9. A hyperplane through the origin with any direction w must satisfy w₁·x₁ + w₂·x₂ = 0 for the boundary — this always passes through (0, 0), far from where the data lives, so it cannot correctly separate the clusters regardless of how w is chosen. LIBSVM would achieve near-perfect accuracy by learning b ≈ −9; the offset-free SVM would be stuck with ~50% accuracy. What makes this scenario interesting is that it does not require any unusual data — it just requires that the practitioner forgets (or is unable) to apply StandardScaler before training. This could happen in real deployment when a pre-trained model encounters data from a different source (a different hospital, a different sensor calibration) that has a systematic mean shift from the training distribution.